In [1]:
import torch.nn as nn

# Section 2: Byte-Pair Encoding (BPE) Tokenizer

## 2.1 The Unicode Standard

In [2]:
ord("牛")

29275

In [4]:
chr(29275)

'牛'

In [5]:
chr(0)

'\x00'

In [6]:
chr(0)

'\x00'

In [8]:
print(chr(0))

 


In [9]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [10]:
print("this is a test" + chr(0) + "string")

this is a test string


In [11]:
len(chr(0))

1

In [12]:
len("")

0

In [12]:
testing_string = "牛"

## 2.2 Unicode Encoding

In [13]:
a = testing_string.encode("utf-8")
b = testing_string.encode("utf-16-be")
c = testing_string.encode("utf-32-be")

In [14]:
print(a)
print(b)
print(c)

b'\xe7\x89\x9b'
b'r['
b'\x00\x00r['


In [5]:
print(type(a))

<class 'bytes'>


In [6]:
list(a)

[104,
 101,
 108,
 108,
 111,
 32,
 33,
 32,
 228,
 184,
 128,
 228,
 186,
 140,
 228,
 184,
 137,
 229,
 155,
 155]

In [18]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes): 
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [19]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes): 
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("你妈了逼".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

## 2.3 subword tokenization

In [20]:
a = "hello world"
b = "你好世界"
print(len(a.split()))
print(len(b.split()))
print(a.encode("utf-8"))
print(b.encode("utf-8"))


2
1
b'hello world'
b'\xe4\xbd\xa0\xe5\xa5\xbd\xe4\xb8\x96\xe7\x95\x8c'


let's focus on one sentence: "the cats are unhappy". 

if we split it word by word: 

the | cats | are | unhappy

each words are single word for model, so the vocab size could be huge, since we have banch of english words. 

and also cat and cats, happy and unhappy are basiclly diff words fo models. since their distance could be far

if we split it by byte: 

t|h|e|␣|c|a|t|s|␣|a|r|e|␣|u|n|h|a|p|p|y 

the vocab size will be fixed 256. and there is no unknow words. 

but the cost is the len of sequence is longer than before approx x5. 

and model need to learn which is word.

finally it's the byte-pair tokenizer

保留 256 个字节作为基础，所以仍然不会有 OOV；再把高频的字节串加进词表，让它们各自变成一个 token。以 the the the 为例：

t h e ␣ t h e ␣ t h e     11 个 token
th e ␣ th e ␣ th e        8 个   （把 t h 加入词表）
the ␣ the ␣ the           5 个   （把 th e 加入词表）


词表每多一项，序列就短一些。代价是词表变大：embedding 层和输出层的参数都随 vocab_size 增长，越少见的 token 在训练中出现的次数也越少。所以 vocab_size 是一个需要权衡的超参数。

BPE 决定加哪些 token

BPE 原本是一种压缩算法（Gage）：反复找出现次数最多的相邻对，用一个新符号替换。Sennrich 等人把它用来切分子词。这个作业实现的是 byte-level BPE：基础单位是字节，每次合并产生一个更长的字节串。这里的"训练"只是在语料上统计频率，决定合并哪些对，不涉及梯度。

一句话总结：子词 tokenizer 用更大的词表换更短的序列，BPE 按出现频率决定把哪些字节串收进词表。

In [24]:
a = bytes([0])
print(a)

b = bytes([1])
print(b)

b'\x00'
b'\x01'


## BPE Tokenizer Training

### Vocabulary initialzation

In [ ]:
def init_vocab(special_tokens: list[str]) -> dict[int, bytes]:
    # special tokens come from the caller; the 256 single bytes are always the same

    vocab = {}
    # for 0 to 255
    for i in range(256):
        vocab[i] = bytes([i])

    for j, token in enumerate(special_tokens):
        vocab[j + 256] = token.encode("utf-8")

    return vocab

In [ ]:
for special_tokens in (["<|endoftext|>"], [], ["<|endoftext|>", "<|pad|>"]):
    vocab = init_vocab(special_tokens)
    n = 256 + len(special_tokens)

    assert len(vocab) == n
    assert set(vocab.keys()) == set(range(n))                 # IDs are 0..n-1
    assert all(isinstance(v, bytes) for v in vocab.values())  # bytes, not int
    assert len(set(vocab.values())) == n                      # no duplicates
    assert sum(len(v) == 1 for v in vocab.values()) == 256    # exactly 256 single bytes
    assert b"a" in vocab.values() and b"\xff" in vocab.values()
    for j, token in enumerate(special_tokens):
        assert vocab[256 + j] == token.encode("utf-8")        # special tokens follow the bytes

print("all checks passed")

### Pre-tokenization

In [30]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [31]:
re.findall(PAT, "dog! dog.")

['dog', '!', ' dog', '.']

In [32]:
re.findall(PAT, "low low low lower")

['low', ' low', ' low', ' lower']

In [33]:
re.findall(PAT, "hello world\n")

['hello', ' world', '\n']

In [34]:
re.findall(PAT, "I have 123 apples!")

['I', ' have', ' 123', ' apples', '!']

In [58]:
a = "low".encode("utf-8")
print(a)
result = []
for i in a:
    result.append(bytes([i]))

result = tuple(result)
print(result)

b'low'
(b'l', b'o', b'w')


In [59]:
b = " low".encode("utf-8")
print(b)
result_1 = []
for i in b:
    result_1.append(bytes([i]))

result_1 = tuple(result_1)
print(result_1)

b' low'
(b' ', b'l', b'o', b'w')


In [60]:
result == result_1

False

In [ ]:
# what is the input? a string list? 
# "low low low lower"

# what should we do? we need to re it first

a = "low low low lower"
b = re.findall(PAT, a)
print(b)

# for each one we change it to ???

counts = {}
for i in b:
    # print(type(i))
    i = i.encode("utf-8")
    middle = []
    for j in i:
        middle.append(bytes([j]))
    middle = tuple(middle)
    # print(middle)
    counts[middle] = counts.get(middle, 0) + 1 

['low', ' low', ' low', ' lower']


In [85]:
def count_pretokens(text: str) -> dict[tuple[bytes, ...], int]:
    pretokens = re.findall(PAT, text)

    counts = {}
    for pretoken in pretokens:
        pretoken_bytes = pretoken.encode("utf-8")
        tokens = []
        for byte_value in pretoken_bytes:
            tokens.append(bytes([byte_value]))
        tokens = tuple(tokens)
        counts[tokens] = counts.get(tokens, 0) + 1

    return counts

In [86]:
counts = count_pretokens("low low low lower")
assert counts == {
    (b'l', b'o', b'w'): 1,
    (b' ', b'l', b'o', b'w'): 2,
    (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
}

counts = count_pretokens("dog! dog.")
assert counts == {
    (b'd', b'o', b'g'): 1,
    (b'!',): 1,
    (b' ', b'd', b'o', b'g'): 1,
    (b'.',): 1,
}
assert all(isinstance(t, bytes) and len(t) == 1 for key in counts for t in key)  # single bytes, not int or str

text = "some text that i'll pre-tokenize"
counts = count_pretokens(text)
assert sum(len(k) * v for k, v in counts.items()) == len(text.encode("utf-8"))  # every byte is covered once

assert count_pretokens("") == {}

print("all checks passed")

all checks passed


In [88]:
tokens = ["<|endoftext|>", "<|pad|>"]
escaped = [re.escape(t) for t in tokens]

pattern = "|".join(escaped)

re.split(pattern, "a<|endoftext|>b<|pad|>c")

['a', 'b', 'c']

In [89]:
text = (
    "Once upon a time, there was a little dog named Max. Max loved to run in the park.\n"
    "One day, he found a red ball. \"This is my ball!\" he said.<|endoftext|>"
    "Lily had 3 cats. She didn't know which one was the best, so she loved them all.<|endoftext|>"
    "<|endoftext|>"
    "The sun was hot. Tom and Sue went to the lake to swim.\n"
    "<|endoftext|>"
)

In [92]:
a = re.split(pattern, text)
print(a)

['Once upon a time, there was a little dog named Max. Max loved to run in the park.\nOne day, he found a red ball. "This is my ball!" he said.', "Lily had 3 cats. She didn't know which one was the best, so she loved them all.", '', 'The sun was hot. Tom and Sue went to the lake to swim.\n', '']


In [93]:
len(a)

5

In [100]:
x = []
for i in a:
    x.append(count_pretokens(i))

print(x)


total = {} 
for i in a:
    counts = count_pretokens(i)
    for key, count in counts.items():
        total[key] = total.get(key, 0) + count

[{(b'O', b'n', b'c', b'e'): 1, (b' ', b'u', b'p', b'o', b'n'): 1, (b' ', b'a'): 3, (b' ', b't', b'i', b'm', b'e'): 1, (b',',): 2, (b' ', b't', b'h', b'e', b'r', b'e'): 1, (b' ', b'w', b'a', b's'): 1, (b' ', b'l', b'i', b't', b't', b'l', b'e'): 1, (b' ', b'd', b'o', b'g'): 1, (b' ', b'n', b'a', b'm', b'e', b'd'): 1, (b' ', b'M', b'a', b'x'): 2, (b'.',): 4, (b' ', b'l', b'o', b'v', b'e', b'd'): 1, (b' ', b't', b'o'): 1, (b' ', b'r', b'u', b'n'): 1, (b' ', b'i', b'n'): 1, (b' ', b't', b'h', b'e'): 1, (b' ', b'p', b'a', b'r', b'k'): 1, (b'\n',): 1, (b'O', b'n', b'e'): 1, (b' ', b'd', b'a', b'y'): 1, (b' ', b'h', b'e'): 2, (b' ', b'f', b'o', b'u', b'n', b'd'): 1, (b' ', b'r', b'e', b'd'): 1, (b' ', b'b', b'a', b'l', b'l'): 2, (b' ', b'"'): 1, (b'T', b'h', b'i', b's'): 1, (b' ', b'i', b's'): 1, (b' ', b'm', b'y'): 1, (b'!', b'"'): 1, (b' ', b's', b'a', b'i', b'd'): 1}, {(b'L', b'i', b'l', b'y'): 1, (b' ', b'h', b'a', b'd'): 1, (b' ', b'3'): 1, (b' ', b'c', b'a', b't', b's'): 1, (b'.',): 2, (

In [ ]:
escaped = [re.escape(t) for t in tokens]

pattern = "|".join(escaped)

re.split(pattern, "a<|endoftext|>b<|pad|>c")

In [112]:
def count_all_pretokens(text, special_tokens):
    if len(special_tokens) != 0:
        escaped = [re.escape(t) for t in special_tokens]
        pattern = "|".join(escaped)
        split_text = re.split(pattern, text)
    else:
        split_text = [text]

    total = {}
    for per_text in split_text:
        counts = count_pretokens(per_text)
        for key, count in counts.items():
            total[key] = total.get(key, 0) + count

    return total

In [113]:
EOT = "<|endoftext|>"

# no merge across a document boundary: only 'a' and 'b', nothing from the special token
assert count_all_pretokens("a<|endoftext|>b", [EOT]) == {(b'a',): 1, (b'b',): 1}

# each document starts fresh: 'low' after the boundary has no leading space
assert count_all_pretokens("low low<|endoftext|>low lower", [EOT]) == {
    (b'l', b'o', b'w'): 2,
    (b' ', b'l', b'o', b'w'): 1,
    (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
}

# several special tokens at once
assert count_all_pretokens("a<|pad|>b<|endoftext|>c", [EOT, "<|pad|>"]) == {(b'a',): 1, (b'b',): 1, (b'c',): 1}

# no special tokens: same as pre-tokenizing the whole text
assert count_all_pretokens("low low lower", []) == count_pretokens("low low lower")

# edge cases: empty text, consecutive and trailing special tokens
assert count_all_pretokens("", [EOT]) == {}
assert count_all_pretokens("a<|endoftext|><|endoftext|>b<|endoftext|>", [EOT]) == {(b'a',): 1, (b'b',): 1}

# special-token bytes never reach the counts, and every other byte is covered once
docs = "Max ran.\n<|endoftext|>Lily didn't.<|endoftext|>"
counts = count_all_pretokens(docs, [EOT])
assert all(b'<' not in key and b'|' not in key for key in counts)
assert sum(len(k) * v for k, v in counts.items()) == len(docs.encode("utf-8")) - docs.count(EOT) * len(EOT)

print("all checks passed")

all checks passed
